In [15]:
import sys

In [16]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive

    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount("/content/drive")
    sys.path.append("/content/drive/MyDrive/structure-loss-classification/")
    my_local_data = "/content/drive/MyDrive/types/"
    %cd '/content/drive/MyDrive/structure-loss-classification/'
    %pip install scienceplots
    %pip install pytorch_lightning
else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"
    my_local_data_struct = "/mnt/g/My Drive/types-struct/"

In [ ]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
import pytorch_lightning as pl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "notebook", "grid"])

In [ ]:
import pickle

In [ ]:
from models.models import LeNet5
from lightning_modules.lightning_modules import LitLeNet5
from visualization.filters import display_filters
from datasets.data_modules import CustomImageDataModule
from train.train import get_features, train_model

In [ ]:
toTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)
aux_data = datasets.ImageFolder(root=my_local_data, transform=toTensorAndNormalize)

In [ ]:
from datasets.datasets import CustomImageFolder

In [ ]:
aux_data_struct = CustomImageFolder(
    root=my_local_data_struct,
    classification_mode="binary",
    transform=toTensorAndNormalize,
)

In [ ]:
# Try to load cached targets first
try:
    with open(f"logdir/cached_targets_struct.pkl", "rb") as f:
        targets = pickle.load(f)
except FileNotFoundError:
    targets = [t for _, t in aux_data_struct]
    # Cache the targets for next time
    with open(f"logdir/cached_targets_struct.pkl", "wb") as f:
        pickle.dump(targets, f)

In [ ]:
model2 = LitLeNet5(num_classes=2, size_layer_1=10, size_layer_2=5, learning_rate=0.001)

In [ ]:
dataset = datasets.ImageFolder(my_local_data_struct)
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class to index mapping: {dataset.class_to_idx}")

In [ ]:
kfold = StratifiedKFold(
    n_splits=4,
    shuffle=True,
)

In [ ]:
validation_metrics = []

In [ ]:
trainer_config = {
    "patience": 3,
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 10,
    "precision": 32,
    "n_steps": 5,
}

In [ ]:
# Assuming aux_data is a dataset object and targets are the labels
train_idx, val_idx, _, _ = train_test_split(
    range(len(aux_data_struct)), targets, test_size=0.2, random_state=42
)

train_data = torch.utils.data.Subset(aux_data_struct, train_idx)
val_data = torch.utils.data.Subset(aux_data_struct, val_idx)

In [ ]:
data_module = CustomImageDataModule(
    train_dataset=train_data,
    val_dataset=val_data,
    batch_size=16,
    num_workers=4,
)

In [ ]:
# Assuming model2 is initialized, and trainer_config is defined
val_metrics = train_model(
    model=model2,
    trainer_config=trainer_config,
    save_dir="logdir-struct/",
    data_module=data_module,
)